# arg-position-back-functions — worked example 3: Register mul_back0 and mul_back1 at both argnums

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `arg-position-back-functions`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Multiplication `out = x * y` is the symmetric counterpart to division: `mul_back0` and `mul_back1` have *swapped* bodies (`grad_out * y` vs `grad_out * x`), not identical ones. Even when two back fns look similar you must register one per `(fwd_fn, argnum)` key so the reverse pass can dispatch by position. This worked example builds the tiny registry and dispatches it.

## Worked solution

**Step 1 — the per-arg derivatives.** For `out = x * y`: `dout/dx = y` and `dout/dy = x`. So `mul_back0 = grad_out * y` (uses arg 1!) and `mul_back1 = grad_out * x` (uses arg 0!). The cross-dependence is the asymmetry — each gradient depends on the *other* input.

**Step 2 — the registry.** `BackFuncs` keys a dict by the tuple `(fwd_fn, argnum)`. `add_back_func` stores; `get_back_func` reads (letting a missing key raise `KeyError` naturally).

**Step 3 — register at BOTH positions.** We register `mul_back0` at `(t.multiply, 0)` and `mul_back1` at `(t.multiply, 1)`. This is the key teaching point: every argnum the forward consumed needs its own entry, even for symmetric-looking ops.

**Step 4 — dispatch and verify.** The reverse pass looks up `(t.multiply, 0)` and `(t.multiply, 1)` and calls each with the uniform signature. We confirm the recovered grads match autograd.

In [ ]:
class BackFuncs:
    def __init__(self):
        self._registry = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._registry[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._registry[(fwd_fn, argnum)]


def mul_back0(grad_out, out, x, y):
    return grad_out * y


def mul_back1(grad_out, out, x, y):
    return grad_out * x


bf = BackFuncs()
bf.add_back_func(t.multiply, 0, mul_back0)
bf.add_back_func(t.multiply, 1, mul_back1)

t.manual_seed(0)
x = t.randn(3, requires_grad=True)
y = t.randn(3, requires_grad=True)
out = x * y
grad_out = t.randn(3)
out.backward(grad_out)

gx = bf.get_back_func(t.multiply, 0)(grad_out, out.detach(), x.detach(), y.detach())
gy = bf.get_back_func(t.multiply, 1)(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))